## Loading Dataset

In [20]:
import pandas as pd
import numpy as np

# Loading the messy data
df = pd.read_csv("../data/retail_store_sales.csv")

print(f"Loaded Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded Dataset: 12575 rows, 11 columns


In [21]:
df

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,NaN
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,NaN
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


## Header Cleaning

In [22]:
print(df.columns.tolist())

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Fix Applied")
print(df.columns.tolist())


['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied']
Fix Applied
['transaction_id', 'customer_id', 'category', 'item', 'price_per_unit', 'quantity', 'total_spent', 'payment_method', 'location', 'transaction_date', 'discount_applied']


## Checking missing values

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    12575 non-null  str    
 1   customer_id       12575 non-null  str    
 2   category          12575 non-null  str    
 3   item              11362 non-null  str    
 4   price_per_unit    11966 non-null  float64
 5   quantity          11971 non-null  float64
 6   total_spent       11971 non-null  float64
 7   payment_method    12575 non-null  str    
 8   location          12575 non-null  str    
 9   transaction_date  12575 non-null  str    
 10  discount_applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.1+ MB


In [24]:
df.isnull().sum()

transaction_id         0
customer_id            0
category               0
item                1213
price_per_unit       609
quantity             604
total_spent          604
payment_method         0
location               0
transaction_date       0
discount_applied    4199
dtype: int64

### Handling Missing Values

In [25]:
# Categorical columns
cat_cols = ['category', 'item', 'payment_method', 'location', 'discount_applied']

for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

In [26]:
# Numerical columns
num_cols = ['quantity', 'price_per_unit']

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    
df['total_spent'] = df['total_spent'].fillna(
    df['quantity'] * df['price_per_unit']
)

In [27]:
df

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,Unknown
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False
...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,Unknown
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,Unknown
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True


In [28]:
print(df.isnull().sum())

transaction_id      0
customer_id         0
category            0
item                0
price_per_unit      0
quantity            0
total_spent         0
payment_method      0
location            0
transaction_date    0
discount_applied    0
dtype: int64


### Checking duplicates

In [29]:
df.duplicated().sum()

np.int64(0)

### Checking text values

In [30]:
df[['category', 'item', 'payment_method', 'location', 'discount_applied']].nunique()

category              8
item                201
payment_method        3
location              2
discount_applied      3
dtype: int64

In [31]:
for col in ['category', 'item', 'payment_method', 'location', 'discount_applied']:
    print(f"\n{col}:")
    print(df[col].unique())


category:
<StringArray>
[                        'Patisserie',                      'Milk Products',
                           'Butchers',                          'Beverages',
                               'Food',                          'Furniture',
      'Electric household essentials', 'Computers and electric accessories']
Length: 8, dtype: str

item:
<StringArray>
[ 'Item_10_PAT', 'Item_17_MILK',  'Item_12_BUT',  'Item_16_BEV',
  'Item_6_FOOD',      'Unknown',  'Item_1_FOOD',  'Item_16_FUR',
  'Item_22_BUT',   'Item_3_BUT',
 ...
   'Item_3_PAT',  'Item_13_FUR',  'Item_18_PAT', 'Item_12_MILK',
 'Item_10_MILK', 'Item_16_FOOD', 'Item_19_FOOD',  'Item_14_EHE',
   'Item_3_EHE',   'Item_9_BUT']
Length: 201, dtype: str

payment_method:
<StringArray>
['Digital Wallet', 'Credit Card', 'Cash']
Length: 3, dtype: str

location:
<StringArray>
['Online', 'In-store']
Length: 2, dtype: str

discount_applied:
[True False 'Unknown']


### Checking Data types

In [32]:
df.dtypes

transaction_id          str
customer_id             str
category                str
item                    str
price_per_unit      float64
quantity            float64
total_spent         float64
payment_method          str
location                str
transaction_date        str
discount_applied     object
dtype: object

In [33]:
df['quantity'] = df['quantity'].astype(int)

In [34]:
df['discount_applied'] = df['discount_applied'].astype(str)

In [35]:
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

In [36]:
df.describe()

,price_per_unit,quantity,total_spent,transaction_date
count,12575.000000,12575.000000,12575.000000,12575
mean,23.348191,5.558648,130.208111,2023-07-12 20:23:41.105367
min,5.000000,1.000000,5.000000,2022-01-01 00:00:00
25%,14.000000,3.000000,52.000000,2022-09-30 00:00:00
50%,23.000000,6.000000,110.000000,2023-07-13 00:00:00
75%,32.000000,8.000000,192.000000,2024-04-24 00:00:00
max,41.000000,10.000000,410.000000,2025-01-18 00:00:00
std,10.480413,2.790160,93.580667,NaN


### Verifying business logic

In [37]:
df['calculated_total'] = df['quantity'] * df['price_per_unit']
df[df['calculated_total'] != df['total_spent']]

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,calculated_total
5,TXN_7482416,CUST_09,Patisserie,Unknown,23.0,10,200.0,Credit Card,Online,2023-11-30,Unknown,230.0
11,TXN_5422631,CUST_09,Milk Products,Unknown,23.0,8,52.0,Digital Wallet,In-store,2025-01-12,True,184.0
17,TXN_9634894,CUST_15,Milk Products,Unknown,23.0,10,275.0,Digital Wallet,Online,2022-04-17,Unknown,230.0
21,TXN_8685338,CUST_15,Milk Products,Unknown,23.0,3,105.0,Credit Card,In-store,2023-10-29,Unknown,69.0
32,TXN_1543244,CUST_20,Food,Unknown,23.0,8,196.0,Credit Card,Online,2024-10-25,True,184.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12435,TXN_5147764,CUST_01,Furniture,Unknown,23.0,5,32.5,Credit Card,Online,2023-09-06,True,115.0
12457,TXN_1352194,CUST_17,Electric household essentials,Unknown,23.0,4,86.0,Credit Card,Online,2023-02-26,Unknown,92.0
12477,TXN_5625684,CUST_22,Computers and electric accessories,Unknown,23.0,4,80.0,Cash,In-store,2022-11-09,True,92.0
12491,TXN_7894525,CUST_23,Butchers,Unknown,23.0,1,26.0,Credit Card,Online,2023-01-31,True,23.0


Missing values were identified in price_per_unit. Missing prices were imputed using the median value (23.0). As a result, business-rule validation (quantity × price_per_unit = total_spent) was performed only on records with complete original pricing information, since imputed values can distort transaction totals.

In [38]:
raw_df = pd.read_csv("../data/retail_store_sales.csv")
raw_df['Price Per Unit'].isnull().sum()

np.int64(609)

In [39]:
valid_rows = df.loc[
    raw_df['Price Per Unit'].notna() & 
    raw_df['Total Spent'].notna()
    ].copy()

valid_rows['calculated_total'] = (valid_rows['quantity'] * valid_rows['price_per_unit'])

incorrect_rows = valid_rows[
    np.abs(valid_rows['calculated_total'] - valid_rows['total_spent']) > 0.01
]

print(f"Incorrect rows: {len(incorrect_rows)}")

incorrect_rows.head()

Incorrect rows: 0


,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,calculated_total


In [40]:
df.drop(columns=['calculated_total'], inplace=True)

## Checking Outliers

In [41]:
Q1 = df['total_spent'].quantile(0.25)
Q3 = df['total_spent'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df['total_spent'] < lower) |
    (df['total_spent'] > upper)
]

print(f"Outliers in Total Spent: {len(outliers)}")

Outliers in Total Spent: 60


In [42]:
outliers['total_spent'].describe()

count     60.0
mean     410.0
std        0.0
min      410.0
25%      410.0
50%      410.0
75%      410.0
max      410.0
Name: total_spent, dtype: float64

In [43]:
df['total_spent'].value_counts().sort_index(ascending=False).head(10)

total_spent
410.0    60
395.0    39
380.0    58
369.0    55
365.0    42
355.5    44
350.0    37
342.0    49
335.0    71
328.5    49
Name: count, dtype: int64

In [44]:
df[df['total_spent'] == 410][
    ['category', 'item', 'quantity', 'price_per_unit', 'total_spent']
].head(20)

,category,item,quantity,price_per_unit,total_spent
27,Furniture,Item_25_FUR,10,41.0,410.0
133,Furniture,Item_25_FUR,10,41.0,410.0
339,Food,Item_25_FOOD,10,41.0,410.0
869,Beverages,Item_25_BEV,10,41.0,410.0
1060,Furniture,Item_25_FUR,10,41.0,410.0
1088,Beverages,Item_25_BEV,10,41.0,410.0
1468,Butchers,Item_25_BUT,10,41.0,410.0
1505,Electric household essentials,Item_25_EHE,10,41.0,410.0
1568,Butchers,Item_25_BUT,10,41.0,410.0
1950,Furniture,Item_25_FUR,10,41.0,410.0


In [45]:
outliers = outliers.copy()  # To avoid SettingWithCopyWarning
outliers['calculated_total'] = (
    outliers['quantity'] * outliers['price_per_unit']
)

incorrect_rows = outliers[
    np.abs(outliers['calculated_total'] - outliers['total_spent']) > 0.01
]
incorrect_rows

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,calculated_total
2153,TXN_5338814,CUST_06,Milk Products,Unknown,23.0,10,410.0,Digital Wallet,Online,2023-03-13,False,230.0
5745,TXN_3855536,CUST_04,Computers and electric accessories,Unknown,23.0,10,410.0,Credit Card,In-store,2024-09-27,True,230.0
8508,TXN_4771109,CUST_21,Electric household essentials,Unknown,23.0,10,410.0,Digital Wallet,In-store,2022-07-03,True,230.0
8596,TXN_2288041,CUST_24,Food,Unknown,23.0,10,410.0,Digital Wallet,In-store,2023-09-02,False,230.0


## Business Logic Validation Findings

- Four transactions violated the rule:

- Total Spent = Quantity × Price Per Unit

- Investigation showed that all four records above had missing Item and Price Per Unit values in the original dataset. Median imputation (23.0) was applied to Price Per Unit, which likely caused the discrepancy. The records were retained but flagged as potentially inaccurate.

In [46]:
# Create a flag column
df['business_rule_valid'] = True

# Flag the problematic rows
df.loc[incorrect_rows.index, 'business_rule_valid'] = False

In [47]:
df['business_rule_valid'].value_counts()

business_rule_valid
True     12571
False        4
Name: count, dtype: int64

In [48]:
df

,transaction_id,customer_id,category,item,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,business_rule_valid
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10,185.0,Digital Wallet,Online,2024-04-08,True,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9,261.0,Digital Wallet,Online,2023-07-23,True,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2,43.0,Credit Card,Online,2022-10-05,False,True
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9,247.5,Credit Card,Online,2022-05-07,Unknown,True
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7,87.5,Digital Wallet,Online,2022-10-02,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
12570,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4,152.0,Credit Card,In-store,2023-09-03,Unknown,True
12571,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9,58.5,Cash,Online,2022-08-12,False,True
12572,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10,140.0,Cash,Online,2024-08-24,Unknown,True
12573,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6,84.0,Cash,Online,2023-12-30,True,True


After cleaning and validation, 99.97% of transactions satisfied the business rule. Four records were flagged for potential data quality issues.

## Saving Cleaned Data

In [49]:
df.to_csv("../data/retail_store_sales_cleaned.csv", index=False)